# Spark Streaming com Kafka e Análise de Sentimentos

In [1]:
KAFKA_BOOTSTRAP = "kafka:9092"
INPUT_TOPIC = "input_topic"
OUTPUT_TOPIC = "output_topic"

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("reddit-sentiment-stream")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-06ff7a26-5276-4f09-b5be-5af70573af62;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0

## Lendo stream do Kafka (topic: input_topic)

In [3]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("post_title", StringType(), True),
    StructField("post_text", StringType(), True),
    StructField("comment", StringType(), True),
])

df = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", INPUT_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

json_df = df.selectExpr("CAST(value AS STRING) as json_str")
parsed_schema = json_df.select(from_json(col("json_str"), schema).alias("data")).select("data.*")
parsed_schema.printSchema()


root
 |-- post_title: string (nullable = true)
 |-- post_text: string (nullable = true)
 |-- comment: string (nullable = true)



## Análise de sentimentos com Vader

In [6]:
import json
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def sentiment_json(text):
    if not text:
        return json.dumps({"neg":0.0,"neu":1.0,"pos":0.0,"compound":0.0})
    try:
        scores = analyzer.polarity_scores(text)
    except Exception:
        scores = {"neg":0.0,"neu":1.0,"pos":0.0,"compound":0.0}
    return json.dumps(scores)

sentiment_udf = udf(sentiment_json, StringType())

with_sentiment = parsed_schema.withColumn("sentiment", sentiment_udf(col("comment")))

## Enviando stream processada ao Kafka (topic: output_topic)

In [7]:
from pyspark.sql.functions import to_json, struct

out_df = with_sentiment.selectExpr("to_json(struct(*)) AS value")

query = (
    out_df.writeStream
          .format("kafka")
          .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
          .option("topic", OUTPUT_TOPIC)
          .option("checkpointLocation", "/tmp/spark_sentiment_checkpoint")
          .outputMode("append")
          .start()
)

25/11/15 14:35:08 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

#### Rode a célula abaixo para finalizar o Spark Streaming

In [13]:
query.stop()

25/11/15 13:25:55 WARN DAGScheduler: Failed to cancel job group b7006d81-bdd4-4e68-b1fd-f17a08098ae2. Cannot find active jobs for it.
25/11/15 13:25:55 WARN DAGScheduler: Failed to cancel job group b7006d81-bdd4-4e68-b1fd-f17a08098ae2. Cannot find active jobs for it.
